# Workflow Evaluation

There would be 30 prompts in a form of csv where there are these combinations

| | categories | prompt styles | stocks |
|---|---:|---:|---:|
| Combinations | 5 | 7 | 5 |
| **Total prompts** | | | **175** |

## Import Libraries

In [ ]:
from collections import defaultdict
import json
from pathlib import Path

from deepeval import evaluate
from deepeval.metrics import (
    TaskCompletionMetric,       # Did the crew actually finish the job correctly (uses tools_called + actual_output)
    ArgumentCorrectnessMetric,  # Were tool call arguments correct given the user input (uses tools_called)
    AnswerRelevancyMetric,      # Is the report relevant to the user prompt (input vs actual_output)
    HallucinationMetric,        # Is the report grounded in the gathered context (context vs actual_output)
)
from deepeval.test_case import LLMTestCase, ToolCall

import csv
import sys

PROJECT_PATH = Path("..").resolve()
sys.path.append(str(PROJECT_PATH))

# import the workflows
from utils.workflows.workflow_A import workflow_A_run as workflow_A
from utils.workflows.workflow_B import workflow_B_run as workflow_B
from utils.workflows.workflow_C import workflow_C_run as workflow_C

In [ ]:
# Wrap the real tools so we can record every call (name + arguments) during a run.
# The original tools are untouched; we only swap the underlying .func on a copy.

import copy as _copy
from utils.tools.search_tool import search_full_articles, search_searxng
from utils.workflows.workflow_A import sentiment_tool as _sentiment_tool_A
from utils.workflows.workflow_B import sentiment_tool as _sentiment_tool_B
from utils.workflows.workflow_C import sentiment_tool as _sentiment_tool_C
from utils.tools.markdown_render_tool import markdown_render_tool

TOOL_CALL_LOG = defaultdict(list)

def spy_tool(tool):
    """Return a copy of `tool` that records every call into TOOL_CALL_LOG."""
    name = tool.name
    original = tool.func

    def wrapper(*args, **kwargs):
        TOOL_CALL_LOG[name].append({"args": args, "kwargs": kwargs})
        return original(*args, **kwargs)

    return tool.model_copy(update={"func": wrapper})

SEARCH_TOOL = spy_tool(search_searxng)
SENTIMENT_TOOL_A = spy_tool(_sentiment_tool_A)
SENTIMENT_TOOL_B = spy_tool(_sentiment_tool_B)
SENTIMENT_TOOL_C = spy_tool(_sentiment_tool_C)
MARKDOWN_TOOL = spy_tool(markdown_render_tool)

In [ ]:
# Load the 175-prompt test matrix
with open(PROJECT_PATH / "prompts_test_matrix.csv") as f:
    prompt_rows = list(csv.DictReader(f))

print(f"{len(prompt_rows)} prompts loaded")

## Evaluation Design

Each workflow run produces one `LLMTestCase` per prompt. The four metrics plug into
different fields of the same test case:

| Metric | Reads | What it measures here |
|---|---|---|
| `TaskCompletionMetric` | `input`, `actual_output`, `tools_called` | Was the workflow's final report a correct completion of the user's request |
| `ArgumentCorrectnessMetric` | `input`, `tools_called` | Did the agents pass sensible arguments to their tools (e.g. the stock ticker in the search query) |
| `AnswerRelevancyMetric` | `input`, `actual_output` | Does the final report actually answer the user prompt |
| `HallucinationMetric` | `context`, `actual_output` | Is every claim in the report traceable to gathered news articles |

`context` is the tool output we captured with the spy wrappers (search results +
sentiment outputs), i.e. the source material the report was written from.

In [ ]:
async def run_workflow(workflow, topic: str):
    """Run one workflow, capturing the final report and tool calls."""
    TOOL_CALL_LOG.clear()
    result = await workflow(topic)
    return str(result), _snapshot_calls()


def _snapshot_calls():
    return dict(TOOL_CALL_LOG)


def make_tool_calls(snapshot: dict) -> list[ToolCall]:
    """Convert recorded calls into deepeval ToolCall objects."""
    tool_calls = []
    for name, calls in snapshot.items():
        for call in calls:
            params = call["kwargs"] or {"input": call["args"][0] if call["args"] else ""}
            tool_calls.append(
                ToolCall(name=name, description="", input_parameters=params)
            )
    return tool_calls

In [ ]:
def build_test_case(prompt: str, report: str, snapshot: dict) -> LLMTestCase:
    tool_calls = make_tool_calls(snapshot)

    # context = everything the report was grounded in (search output + sentiment outputs)
    context = []
    for name, calls in snapshot.items():
        for call in calls:
            context.append(str(call))

    return LLMTestCase(
        input=prompt,
        actual_output=report,
        context=context,
        tools_called=tool_calls,
    )

In [ ]:
task_completion_metric = TaskCompletionMetric(threshold=0.5)
argument_correctness_metric = ArgumentCorrectnessMetric(threshold=0.5)
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.5)
hallucination_metric = HallucinationMetric(threshold=0.5)

metrics = [
    task_completion_metric,
    argument_correctness_metric,
    answer_relevancy_metric,
    hallucination_metric,
]

In [ ]:
# NOTE: 175 prompts x 3 workflows is expensive (LLM + scraping + FinBERT).
# Start small: slice the CSV (e.g. prompt_rows = prompt_rows[:5]) for a smoke test.

In [ ]:
results = []

for row in prompt_rows[:5]:  # start small, remove slice for full run
    prompt = row["prompt"]

    for name, workflow in [("A", workflow_A), ("B", workflow_B), ("C", workflow_C)]:
        report, snapshot = await run_workflow(workflow, prompt)
        test_case = build_test_case(prompt, report, snapshot)
        results.append({"row": row, "workflow": name, "test_case": test_case})
        print(f"[prompt {row['prompt_id']}] workflow {name}: report {len(report)} chars, {len(snapshot)} tool types called")

print(f"{len(results)} test cases created")

In [ ]:
from deepeval import evaluate

test_cases = [r["test_case"] for r in results]

eval_results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    # run in notebook's event loop
    # async_mode defaults on; deepeval handles it
)

# quick per-workflow score summary
summary = defaultdict(list)
for r, er in zip(results, eval_results):
    for m in er.metrics:
        summary[(r["workflow"], m.name)].append(m.score)

for (wf, metric_name), scores in sorted(summary.items()):
    scores = [s for s in scores if s is not None]
    print(f"workflow {wf} | {metric_name:30s} | avg: {sum(scores)/len(scores):.2f} | n: {len(scores)}")

In [ ]:
# Log results to CSV: one row per (prompt, workflow, metric)

import csv
from datetime import datetime

def export_results_csv(results, eval_results, out_path=None) -> str:
    rows = []
    for r, er in zip(results, eval_results):
        m = {mt.name: mt for mt in er.metrics}
        out_row = {
            "prompt_id": r["row"]["prompt_id"],
            "category": r["row"]["category"],
            "stock": r["row"]["stock"],
            "prompt_style": r["row"]["prompt_style"],
            "prompt": r["row"]["prompt"],
            "workflow": r["workflow"],
        }
        for metric in metrics:
            mt = m.get(metric.name)
            out_row[f"{metric.name}_score"] = mt.score if mt else None
            out_row[f"{metric.name}_passed"] = (mt.score >= metric.threshold) if mt and mt.score is not None else None
        rows.append(out_row)

    if out_path is None:
        out_path = PROJECT_PATH / "evaluation_results" / f"eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    Path(out_path).parent.mkdir(exist_ok=True, parents=True)

    fieldnames = list(rows[0].keys())
    with open(out_path, "w", newline="") as f:
        wcsv = csv.DictWriter(f, fieldnames=fieldnames)
        wcsv.writeheader()
        wcsv.writerows(rows)

    return str(out_path)

csv_path = export_results_csv(results, eval_results)
print("results saved to", csv_path)